In [17]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from pathlib import Path

OUT = Path("../data_processed")

pack = pd.read_csv(OUT / "human_eval_pack.csv")         # Day9: 10产品×3版本题库
pref = pd.read_csv(OUT / "human_preference.csv")        # synthetic 或真人偏好（长表）
print("pack:", pack.shape)
print("pref:", pref.shape)
pack.head(3)

pack: (30, 5)
pref: (150, 5)


,product_id,category,variant_id,title,bullets
0,P420652,Skincare,v1,LANEIGE Lip Sleeping Mask Intense Hydration wi...,- like\n- face\n- absorbs quickly\n- great for...
1,P420652,Skincare,v2,LANEIGE Lip Sleeping Mask Intense Hydration wi...,- feel\n- using\n- absorbs quickly\n- great fo...
2,P420652,Skincare,v3,LANEIGE Lip Sleeping Mask Intense Hydration wi...,- skin\n- feel\n- absorbs quickly\n- great for...


In [18]:
cnt = pref.groupby(["product_id", "variant_id"]).size().reset_index(name="cnt")
cnt["total"] = cnt.groupby("product_id")["cnt"].transform("sum")
cnt["win_rate"] = (cnt["cnt"] / cnt["total"]).round(3)

win_rate = cnt.sort_values(["product_id","win_rate"], ascending=[True, False])
winners = win_rate.groupby("product_id").head(1)[["product_id","variant_id","win_rate"]] \
                  .rename(columns={"variant_id":"winner_variant", "win_rate":"winner_win_rate"})

winners.head(10)

,product_id,winner_variant,winner_win_rate
0,P218700,v1,0.533
3,P248407,v1,0.533
6,P269122,v1,0.533
9,P394639,v1,0.533
12,P411387,v1,0.467
16,P417238,v2,0.400
20,P420652,v3,0.533
21,P427421,v1,0.400
25,P450271,v2,0.400
27,P7880,v1,0.333


In [19]:
pack_labeled = pack.merge(winners, on="product_id", how="left")
pack_labeled["is_winner"] = (pack_labeled["variant_id"] == pack_labeled["winner_variant"]).astype(int)

# 拼一个用于分析的文本（title + bullets）
pack_labeled["text"] = pack_labeled["title"].astype(str) + "\n" + pack_labeled["bullets"].astype(str)

pack_labeled[["product_id","variant_id","is_winner","winner_win_rate"]].head(9)

,product_id,variant_id,is_winner,winner_win_rate
0,P420652,v1,0,0.533
1,P420652,v2,0,0.533
2,P420652,v3,1,0.533
3,P7880,v1,1,0.333
4,P7880,v2,0,0.333
5,P7880,v3,0,0.333
6,P218700,v1,1,0.533
7,P218700,v2,0,0.533
8,P218700,v3,0,0.533


In [20]:
def tokenize(text):
    text = str(text).lower()
    return re.findall(r"[a-zA-Z']+", text)

win_tokens = Counter()
lose_tokens = Counter()

for _, r in pack_labeled.iterrows():
    toks = tokenize(r["text"])
    if r["is_winner"] == 1:
        win_tokens.update(toks)
    else:
        lose_tokens.update(toks)

rows = []
all_terms = set(win_tokens) | set(lose_tokens)
for t in all_terms:
    w = win_tokens[t]
    l = lose_tokens[t]
    if w + l < 5:   # 太少的词先忽略
        continue
    rows.append((t, w, l, w - l))

pref_terms = pd.DataFrame(rows, columns=["term","win_cnt","lose_cnt","lift"]) \
             .sort_values("lift", ascending=False)

pref_terms.head(30)

,term,win_cnt,lose_cnt,lift
9,like,12,4,8
12,face,8,5,3
0,cleansing,2,4,-2
24,the,2,4,-2
21,cleanser,2,4,-2
18,makeup,2,4,-2
17,hydration,2,4,-2
14,farmacy,2,4,-2
1,balm,2,4,-2
13,intense,2,4,-2


In [21]:
# 取 top/bottom 作为策略
BOOST = pref_terms.head(20)["term"].tolist()
BLOCK = pref_terms.tail(30)["term"].tolist()

print("BOOST:", BOOST[:12])
print("BLOCK:", BLOCK[:12])

BOOST: ['like', 'face', 'cleansing', 'the', 'cleanser', 'makeup', 'hydration', 'farmacy', 'balm', 'intense', 'green', 'cream']
BLOCK: ['like', 'face', 'cleansing', 'the', 'cleanser', 'makeup', 'hydration', 'farmacy', 'balm', 'intense', 'green', 'cream']


In [22]:
def clean_block(text, block_list):
    words = str(text).split()
    words2 = []
    for w in words:
        if w.lower().strip(".,!?:;") in set(block_list):
            continue
        words2.append(w)
    return " ".join(words2)

def generate_variant_iter1(base_title, base_bullets, boost, block):
    boost0 = boost[0] if len(boost) > 0 else "highlight"
    boost1 = boost[1] if len(boost) > 1 else boost0
    title = str(base_title)
    bullets = str(base_bullets)

    # title：若没包含 boost，则追加一个
    if not any(b in title.lower() for b in boost[:8]):
        title = (title + " | " + boost[0])[:60]

    # bullets：前两条换成 boost 词（保持 5 条不变）
    lines = [x.strip() for x in bullets.split("\n") if x.strip()]
    if len(lines) >= 2:
        lines[0] = "- " + boost[0]
        lines[1] = "- " + (boost[1] if len(boost)>1 else boost[0])
    bullets2 = "\n".join(lines)

    # block 过滤
    title = clean_block(title, block)
    bullets2 = "\n".join([clean_block(x, block) for x in bullets2.split("\n")])

    return title, bullets2

# 基于每个产品的 winner 版本做迭代（更合理）
iter1_rows = []
for pid, sub in pack_labeled.groupby("product_id"):
    w = sub.sort_values("is_winner", ascending=False).iloc[0]  # winner 行
    new_title, new_bullets = generate_variant_iter1(w["title"], w["bullets"], BOOST, BLOCK)

    iter1_rows.append({
        "product_id": pid,
        "base_winner_variant": w["winner_variant"],
        "base_winner_win_rate": w["winner_win_rate"],
        "base_title": w["title"],
        "base_bullets": w["bullets"],
        "new_title": new_title,
        "new_bullets": new_bullets
    })

iter1 = pd.DataFrame(iter1_rows)
iter1.head(5)

,product_id,base_winner_variant,base_winner_win_rate,base_title,base_bullets,new_title,new_bullets
0,P218700,v1,0.533,Josie Maran 100 percent Pure Argan Oil | like,- like\n- face\n- absorbs quickly\n- great for...,Josie Maran 100 percent Pure Argan Oil |,-\n-\n-\n-\n-
1,P248407,v1,0.533,First Aid Beauty Ultra Repair Cream Intense Hy...,- like\n- face\n- absorbs quickly\n- great for...,First Aid Beauty Ultra Repair |,-\n-\n-\n-\n-
2,P269122,v1,0.533,Dr. Dennis Gross Skincare Alpha Beta Extra Str...,- like\n- face\n- absorbs quickly\n- great for...,Dr. Dennis Gross Skincare Alpha Beta Extra Str...,-\n-\n-\n-\n-
3,P394639,v1,0.533,belif The True Cream Aqua Bomb | like,- like\n- face\n- absorbs quickly\n- great for...,belif True Aqua Bomb |,-\n-\n-\n-\n-
4,P411387,v1,0.467,Youth To The People Superfood Antioxidant Clea...,- like\n- face\n- absorbs quickly\n- great for...,Youth To People Superfood Antioxidant |,-\n-\n-\n-\n-


In [23]:
def count_hits(text, terms):
    t = str(text).lower()
    return sum(1 for x in terms if x in t)

iter1["base_boost_hits"]  = (iter1["base_title"] + "\n" + iter1["base_bullets"]).apply(lambda x: count_hits(x, BOOST))
iter1["base_block_hits"]  = (iter1["base_title"] + "\n" + iter1["base_bullets"]).apply(lambda x: count_hits(x, BLOCK))

iter1["new_boost_hits"]   = (iter1["new_title"] + "\n" + iter1["new_bullets"]).apply(lambda x: count_hits(x, BOOST))
iter1["new_block_hits"]   = (iter1["new_title"] + "\n" + iter1["new_bullets"]).apply(lambda x: count_hits(x, BLOCK))

iter1[["product_id","base_winner_win_rate","base_boost_hits","new_boost_hits","base_block_hits","new_block_hits"]]

,product_id,base_winner_win_rate,base_boost_hits,new_boost_hits,base_block_hits,new_block_hits
0,P218700,0.533,8,0,12,0
1,P248407,0.533,11,0,15,0
2,P269122,0.533,8,0,13,1
3,P394639,0.533,10,1,14,1
4,P411387,0.467,11,0,15,0
5,P417238,0.400,13,0,18,0
6,P420652,0.533,8,0,14,0
7,P427421,0.400,8,0,12,0
8,P450271,0.400,13,0,18,0
9,P7880,0.333,10,0,14,0


In [24]:
save_path = OUT / "iteration1_compare_report.csv"
iter1.to_csv(save_path, index=False)
print("saved:", save_path)

saved: ../data_processed/iteration1_compare_report.csv


In [25]:
i = 0
print("BOOST[0:5]:", BOOST[:5])
print("BLOCK[0:5]:", BLOCK[:5])

print("\n--- BASE ---")
print(iter1.loc[i, "base_title"])
print(iter1.loc[i, "base_bullets"])

print("\n--- NEW ---")
print(iter1.loc[i, "new_title"])
print(iter1.loc[i, "new_bullets"])

BOOST[0:5]: ['like', 'face', 'cleansing', 'the', 'cleanser']
BLOCK[0:5]: ['like', 'face', 'cleansing', 'the', 'cleanser']

--- BASE ---
Josie Maran 100 percent Pure Argan Oil | like
- like
- face
- absorbs quickly
- great for daily routine
- patch test if sensitive

--- NEW ---
Josie Maran 100 percent Pure Argan Oil |
-
-
-
-
-


In [26]:
import re

def count_hits_wordlevel(text, terms):
    t = str(text).lower()
    tokens = set(re.findall(r"[a-zA-Z']+", t))
    # 只对单词型 terms 生效（你的 BOOST/BLOCK 目前来自 tokenize，所以是单词）
    return sum(1 for x in terms if x in tokens)

In [27]:
iter1["base_boost_hits"]  = (iter1["base_title"] + "\n" + iter1["base_bullets"]).apply(lambda x: count_hits_wordlevel(x, BOOST))
iter1["base_block_hits"]  = (iter1["base_title"] + "\n" + iter1["base_bullets"]).apply(lambda x: count_hits_wordlevel(x, BLOCK))

iter1["new_boost_hits"]   = (iter1["new_title"] + "\n" + iter1["new_bullets"]).apply(lambda x: count_hits_wordlevel(x, BOOST))
iter1["new_block_hits"]   = (iter1["new_title"] + "\n" + iter1["new_bullets"]).apply(lambda x: count_hits_wordlevel(x, BLOCK))

iter1[["product_id","base_boost_hits","new_boost_hits","base_block_hits","new_block_hits"]]

,product_id,base_boost_hits,new_boost_hits,base_block_hits,new_block_hits
0,P218700,8,0,12,0
1,P248407,11,0,15,0
2,P269122,8,0,12,0
3,P394639,10,0,14,0
4,P411387,10,0,14,0
5,P417238,13,0,18,0
6,P420652,8,0,14,0
7,P427421,8,0,12,0
8,P450271,13,0,18,0
9,P7880,9,0,13,0


In [28]:
delta = (iter1[["base_boost_hits","new_boost_hits","base_block_hits","new_block_hits"]]
         .mean().round(2))
delta

base_boost_hits     9.8
new_boost_hits      0.0
base_block_hits    14.2
new_block_hits      0.0
dtype: float64

In [29]:
print("BOOST:", BOOST[:30])
print("BLOCK:", BLOCK[:30])
print("overlap:", sorted(set(BOOST).intersection(set(BLOCK)))[:30])

BOOST: ['like', 'face', 'cleansing', 'the', 'cleanser', 'makeup', 'hydration', 'farmacy', 'balm', 'intense', 'green', 'cream', 'clean', 'using', 'test', 'if', 'absorbs', 'quickly', 'for', 'sensitive']
BLOCK: ['like', 'face', 'cleansing', 'the', 'cleanser', 'makeup', 'hydration', 'farmacy', 'balm', 'intense', 'green', 'cream', 'clean', 'using', 'test', 'if', 'absorbs', 'quickly', 'for', 'sensitive', 'patch', 'great', 'routine', 'daily', 'skin', 'feel']
overlap: ['absorbs', 'balm', 'clean', 'cleanser', 'cleansing', 'cream', 'face', 'farmacy', 'for', 'green', 'hydration', 'if', 'intense', 'like', 'makeup', 'quickly', 'sensitive', 'test', 'the', 'using']


In [30]:
STOP = {"like","way","used","use","using","did","don","time","really","just","got","get","one",
        "also","still","even","much","very","good","great","love","product","im","ive",
        "would","could","make","made","thing","things","feel","feels","face","skin"}

BOOST2 = [t for t in BOOST if (t not in set(BLOCK)) and (t not in STOP) and (len(t) >= 3)]
print("BOOST2:", BOOST2[:20], "len=", len(BOOST2))

BOOST2: [] len= 0


In [32]:
# ====== Better fallback: build BOOST2 from category-specific keywords ======
STOP = {
    "product","products","use","using","used","make","made","really","just","does","did","day","look",
    "love","like","way","time","feel","feels","face","skin","one","also","still","even","much","very",
    "good","great","im","ive","would","could","thing","things"
}

# 1) 找到本批产品的主类目（winner样本里最多的那个）
main_cat = pack_labeled["category"].value_counts().index[0] if "category" in pack_labeled.columns else None
print("main_cat:", main_cat)

# 2) 读取你 Day3 的 pos candidates（注意你这个文件名是 keyword_lexicon_pos_candidates.csv）
lex_pos = pd.read_csv(OUT / "keyword_lexicon_pos_candidates.csv")

# 3) 只取主类目下的 top 80 词，然后过滤 stopwords/block，最后取前 20
cand = lex_pos[lex_pos["category"] == main_cat].copy()
cand["keyword"] = cand["keyword"].astype(str).str.lower().str.strip()

BOOST2 = (
    cand.sort_values("freq", ascending=False)["keyword"]
        .dropna()
        .tolist()
)

BOOST2 = [t for t in BOOST2 if (t not in STOP) and (t not in set(BLOCK)) and (len(t) >= 3)]
BOOST2 = BOOST2[:20]

# 4) 如果该类目没覆盖（cand为空），再用通用电商词兜底
if len(BOOST2) == 0:
    BOOST2 = ["moisturizing","hydrating","light","gentle","clean","smooth","soft",
              "non greasy","absorbs quickly","easy to use"]

print("BOOST2 len:", len(BOOST2))
print("BOOST2 sample:", BOOST2[:15])

main_cat: Skincare
BOOST2 len: 20
BOOST2 sample: ['oil', 'recommend', 'light', 'moisturizer', 'don', 'smooth', 'night', 'definitely', 'soft', 'doesn', 'try', 'little', 'amazing', 'received', 'long']


In [33]:
STOP_EXTRA = {
    "don","doesn","didn","isn","aren","wasn","weren","wouldn","couldn","shouldn",
    "definitely","amazing","received","try","trying","little","really","just","also","still",
    "one","day","look","make","made","use","using","used","product","products"
}

STOP2 = set(STOP) | STOP_EXTRA
BOOST2_clean = [t for t in BOOST2 if t not in STOP2 and len(t) >= 3]
print("BOOST2_clean len:", len(BOOST2_clean))
print("BOOST2_clean sample:", BOOST2_clean[:15])

BOOST2_clean len: 13
BOOST2_clean sample: ['oil', 'recommend', 'light', 'moisturizer', 'smooth', 'night', 'soft', 'long', 'nice', 'eye', 'feeling', 'super', 'texture']


In [34]:
# 重新生成 iter1（用更干净的 BOOST2_clean）
iter1_rows = []
for pid, sub in pack_labeled.groupby("product_id"):
    w = sub.sort_values("is_winner", ascending=False).iloc[0]
    new_title, new_bullets = generate_variant_iter1(w["title"], w["bullets"], BOOST2_clean, BLOCK)
    iter1_rows.append({
        "product_id": pid,
        "base_title": w["title"], "base_bullets": w["bullets"],
        "new_title": new_title,   "new_bullets": new_bullets,
        "base_winner_win_rate": w["winner_win_rate"],
    })
iter1 = pd.DataFrame(iter1_rows)

In [35]:
def count_hits(text, terms):
    t = str(text).lower()
    return sum(1 for x in terms if x in t)

iter1["base_boost_hits"] = (iter1["base_title"] + "\n" + iter1["base_bullets"]).apply(lambda x: count_hits(x, BOOST2_clean))
iter1["new_boost_hits"]  = (iter1["new_title"]  + "\n" + iter1["new_bullets"]).apply(lambda x: count_hits(x, BOOST2_clean))
iter1["base_block_hits"] = (iter1["base_title"] + "\n" + iter1["base_bullets"]).apply(lambda x: count_hits(x, BLOCK))
iter1["new_block_hits"]  = (iter1["new_title"]  + "\n" + iter1["new_bullets"]).apply(lambda x: count_hits(x, BLOCK))

iter1[["product_id","base_boost_hits","new_boost_hits","base_block_hits","new_block_hits"]], iter1[["base_boost_hits","new_boost_hits","base_block_hits","new_block_hits"]].mean().round(2)

(  product_id  base_boost_hits  new_boost_hits  base_block_hits  new_block_hits
 0    P218700                1               2               12               0
 1    P248407                0               2               15               0
 2    P269122                0               2               13               1
 3    P394639                0               2               14               1
 4    P411387                1               3               15               0
 5    P417238                0               2               18               0
 6    P420652                0               2               14               0
 7    P427421                0               2               12               0
 8    P450271                0               2               18               0
 9      P7880                0               2               14               0,
 base_boost_hits     0.2
 new_boost_hits      2.1
 base_block_hits    14.5
 new_block_hits      0.2
 dtype: float64)